<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Estimate long-term elasticities from simulation price/consumption time series.
- **Primary inputs**: runs/sensitivity_*/output.csv
- **Primary outputs**: elasticity result tables
- **Refactor role**: Keep as dedicated elasticity module; use same run-discovery helper as other post-processing notebooks.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Load sensitivity scenarios from one elasticity run folder.
2. Assemble fuel price and consumption time series by scenario.
3. Estimate long-term elasticity indicators for reporting.

### Inputs
- `runs/sensitivity_<date_time>/*/output.csv`

### Outputs
- Elasticity summary table in notebook output.

### How To Reuse
- Set `folder` to another available run in `elasticity/runs/`.


In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.linear_model import LinearRegression

In [ ]:
# Refactor: elasticity runs are under elasticity/runs
folder = os.path.join('runs', 'sensitivity_20231227_090346')

# Read Prices/Consumption data

In [ ]:
energy = ['Electricity', 'Natural gas', 'Wood fuel']
consumption = ['Consumption {} (TWh)'.format(i) for i in energy]
prices = ['Prices {} (euro/kWh)'.format(i)  for i in energy]

rslt = {}
for f in os.listdir(folder):
    if os.path.isdir(os.path.join(folder, f)):
        file = os.path.join(folder, f, 'output.csv')
        df = pd.read_csv(file, header=[0], index_col=[0])

        c = df.loc[consumption, :].set_axis(energy).stack().reset_index()
        c.columns = ['Energy', 'Year', 'Consumption']

        p = df.loc[prices, :].set_axis(energy).stack().reset_index()
        p.columns = ['Energy', 'Year', 'Prices']

        temp = (p.set_index(['Energy', 'Year']).squeeze() * c.set_index(['Energy', 'Year']).squeeze()).groupby('Year').sum() / c.set_index(['Energy', 'Year']).squeeze().groupby('Year').sum()
        temp = pd.concat([temp.rename('Prices')], keys=['Global'], names=['Energy']).reset_index()
        p = pd.concat((temp, p), axis=0)

        temp = pd.concat([c.groupby('Year').sum()], keys=['Global'], names=['Energy']).reset_index()
        c = pd.concat((temp, c), axis=0)

        df = c.merge(p, on=['Energy', 'Year'])
        rslt.update({f: df.copy()})

rslt = pd.concat(rslt, axis=0).reset_index(0).rename(columns={'level_0': 'Scenario'}).reset_index(drop=True)
rslt = rslt.astype({'Year': int})

# remove 2019 - policies are still there
rslt = rslt[rslt['Year'] > 2019]

# log variables to get better regression and directly elasticity
rslt['Log Consumption'] = np.log(rslt['Consumption'])
rslt['Log Prices'] = np.log(rslt['Prices'])

In [ ]:
rslt

# Calculation of long-term elasticities

In [ ]:
result = {}
for k, i in enumerate(energy + ['Global']):
    fig, ax = plt.subplots(1, 1, figsize=(12.8, 9.6))
    df = rslt[rslt['Energy'] == i]
    sns.lineplot(data=df, x='Prices', y='Consumption', hue='Scenario', ax=ax)
    ax.set_title(i)
    ax.legend(labels=[''])
    regr = LinearRegression()
    regr.fit(df.loc[:, ['Year', 'Log Prices']], df.loc[:, 'Log Consumption'])
    print('{} : {}'.format(i, regr.coef_[1].round(3)))
    plt.show()
    result.update({i: regr.coef_[1].round(3)})

In [ ]:
print(pd.Series(result))